# Hands-on Projects — Vector Databases

**Module:** 02 — Vector Databases

Semantic search, filtered document search, and RAG knowledge base projects.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Build semantic search
- Add hybrid+filters
- Wire RAG with citations
- Eval recall@k


## Project 1: Semantic Search Engine

**Definition.** Embed docs/queries; return top-k.

**Why it matters.** Core knowledge search UX.

**How it works.** Ingest→embed→search→snippets.

**Intuition.** Google for your corpus.

**Common pitfalls.**
- No gold queries

**When to use.** FAQ/note search.

```mermaid
flowchart LR
 U[Query]-->E[Embed]-->S[Top-k]-->UI[Results]
```


In [ ]:
import numpy as np
from dataclasses import dataclass
@dataclass
class Hit:
    id:str; score:float; text:str
CORPUS={'c1':'Reset password in settings.','c2':'Refunds within 30 days.','c3':'Shipping 3-5 days.'}
def emb(t,d=64):
    r=np.random.default_rng(abs(hash(t.lower()))%(2**32)); v=r.normal(size=d).astype('float32'); return v/(np.linalg.norm(v)+1e-9)
class SS:
    def __init__(self,c):
        self.ids=list(c); self.texts=[c[i] for i in self.ids]; self.M=np.stack([emb(t) for t in self.texts])
    def search(self,q,k=3):
        s=self.M@emb(q); i=np.argsort(-s)[:k]; return [Hit(self.ids[j],float(s[j]),self.texts[j]) for j in i]
engine=SS(CORPUS)
for h in engine.search('money back'): print(h)


In [ ]:
gold={'money back':{'c2'}}
for q,rel in gold.items():
    got={h.id for h in engine.search(q,2)}; print(q, len(got&rel)/len(rel))


In [ ]:
print('\n'.join(f'{i}. {h.text}' for i,h in enumerate(engine.search('track order'),1)))


In [ ]:
import numpy as np
np.save('_p1.npy', engine.M); print(np.load('_p1.npy').shape)


### Try it yourself — Project 1: Semantic Search Engine

1. 10 docs + 5 gold; recall@3.


## Project 2: Document Search System

**Definition.** Filters + hybrid BM25/dense + pages.

**Why it matters.** Prod search ≠ vectors only.

**How it works.** Payloads→filter→RRF→page.

**Intuition.** Meaning + keywords + facets.

**Common pitfalls.**
- Tenant bugs

**When to use.** Help centers.


In [ ]:
import math
from collections import Counter
import numpy as np
DOCS=[{'id':'d1','text':'Error E1234 disk full','tenant':'acme','lang':'en'},{'id':'d2','text':'Refund within 30 days','tenant':'acme','lang':'en'},{'id':'d3','text':'Disk cleanup','tenant':'globex','lang':'en'}]
def tok(s): return s.lower().split()
def bm25(q,docs):
    T=[tok(d['text']) for d in docs]; N=len(docs); avg=np.mean([len(t) for t in T]); df=Counter(t for ts in T for t in set(ts)); out=[]
    for toks in T:
        tf=Counter(toks); s=0.0
        for term in tok(q):
            if term in df:
                idf=math.log(1+(N-df[term]+.5)/(df[term]+.5)); s+=idf*(tf[term]*2.5)/(tf[term]+1.5+1e-9)
        out.append(s)
    return np.array(out)
def emb(t,d=32):
    r=np.random.default_rng(abs(hash(t))%(2**32)); v=r.normal(size=d); return v/(np.linalg.norm(v)+1e-9)
def rrf(lists,k=60):
    sc={}
    for lst in lists:
        for r,i in enumerate(lst): sc[i]=sc.get(i,0)+1/(k+r+1)
    return sorted(sc.items(), key=lambda x:-x[1])
def hybrid(q,tenant,lang,k=3):
    cand=[d for d in DOCS if d['tenant']==tenant and d['lang']==lang]
    br=[cand[i]['id'] for i in np.argsort(-bm25(q,cand))]; M=np.stack([emb(d['text']) for d in cand]); dr=[cand[i]['id'] for i in np.argsort(-(M@emb(q)))]
    by={d['id']:d for d in cand}; return [(s,by[i]) for i,s in rrf([br,dr])[:k]]
print(hybrid('disk error E1234','acme','en'))


In [ ]:
print(hybrid('disk error E1234','acme','en',10)[:2])


In [ ]:
def guarded(q,auth,lang):
    assert auth; return hybrid(q,auth,lang)
print(guarded('refund','acme','en'))


In [ ]:
assert all(d['tenant']=='acme' for _,d in hybrid('refund','acme','en')); print('ok')


### Try it yourself — Project 2: Document Search System

1. Add product_area filter.
2. Test catching tenant leakage.


## Project 3: RAG Knowledge Base

**Definition.** Retrieve → pack → (stub) LLM → citations.

**Why it matters.** Grounded private Q&A.

**How it works.** Retrieve k → budget → prompt → answer+cites.

**Intuition.** Open-book exam.

**Common pitfalls.**
- No citations
- Blame LLM for retrieval misses

**When to use.** Support/policy bots.

```mermaid
flowchart TD
 Q-->R[Retrieve]-->P[Pack]-->L[LLM]-->A[Answer+cites]
```


In [ ]:
import numpy as np
from dataclasses import dataclass
@dataclass
class Hit:
    id:str; score:float; text:str
def emb(t,d=64):
    r=np.random.default_rng(abs(hash(t.lower()))%(2**32)); v=r.normal(size=d).astype('float32'); return v/(np.linalg.norm(v)+1e-9)
class SS:
    def __init__(self,c):
        self.ids=list(c); self.texts=[c[i] for i in self.ids]; self.M=np.stack([emb(t) for t in self.texts])
    def search(self,q,k=3):
        s=self.M@emb(q); i=np.argsort(-s)[:k]; return [Hit(self.ids[j],float(s[j]),self.texts[j]) for j in i]
KB=SS({'p1':'PTO accrues 1.5 days/month.','p2':'Invoices due by the 5th.','p3':'Security training yearly.'})
def pack(hits,b=400):
    parts=[]; u=0
    for h in hits:
        s=f'[{h.id}] {h.text}'
        if u+len(s)>b: break
        parts.append(s); u+=len(s)
    return '\n'.join(parts)
def rag(q):
    hits=KB.search(q); ctx=pack(hits); return ctx.splitlines()[0] if ctx else 'none', [h.id for h in hits]
print(rag('How much PTO?'))


In [ ]:
def refuse(q,thr=0.05):
    h=KB.search(q)
    return 'no docs' if not h or h[0].score<thr else rag(q)[0]
print(refuse('How much PTO?'))


In [ ]:
import json
ans,cites=rag('security training'); print(json.dumps({'answer':ans,'citations':cites},indent=2))


In [ ]:
print('adapter: upsert/search — swap Pinecone/Qdrant; KEY=YOUR_API_KEY')


### Try it yourself — Project 3: RAG Knowledge Base

1. Swap stub LLM for API via env key.
2. % answers with citations on 20 Qs.


## Glossary

- **citation**: Source id for an answer
- **recall@k**: Hit rate in top-k


## Operator Checklist

- [ ] model+dim+metric documented
- [ ] collection naming by env/model
- [ ] auth-bound tenant filters
- [ ] recall@k + p95 gates
- [ ] re-embed/delete playbooks


In [ ]:
for i,x in enumerate(['model+dim+metric','naming','tenant filter','recall gate','p95 gate','reembed playbook'],1):
    print(f'{i}. [ ] {x}')


### Try it yourself — Readiness

1. Fill the checklist for a real system.
2. Name the top 6-month risk if ignored.
3. Pick one paging metric.


In [ ]:
def gib(n,dim,b=4,r=2,o=1.5):
    return n*dim*b*r*o/(1024**3)
print(f'{gib(5_000_000,768):.1f} GiB for 5M×768-d')


## End-to-End Flow

```mermaid
flowchart LR
  D[Docs]-->C[Chunk]-->E[Embed]-->U[Upsert]-->I[(Index)]
  Q[Query]-->E2[Embed]-->S[Search+filter]
  I-->S-->A[App/RAG]
```


In [ ]:
def inject_tenant(auth, user=None):
    base={'tenant':{'$eq':auth}}
    return {'$and':[base,user]} if user else base
print(inject_tenant('acme',{'lang':{'$eq':'en'}}))


In [ ]:
import numpy as np
rng=np.random.default_rng(1); X=rng.normal(size=(200,16)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
q=X[0]; idx=np.argsort(-(X@q))[:5]; print(idx.tolist())


In [ ]:
import numpy as np
rng=np.random.default_rng(2); X=rng.normal(size=(200,16)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
q=X[0]; idx=np.argsort(-(X@q))[:5]; print(idx.tolist())


In [ ]:
import numpy as np
rng=np.random.default_rng(3); X=rng.normal(size=(200,16)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
q=X[0]; idx=np.argsort(-(X@q))[:5]; print(idx.tolist())


## Summary & Key Takeaways

- Search→hybrid docs→RAG citations.
- Stable APIs ease vendor swaps.
- Eval retrieval first.

### Practice

Ship three projects with shared embed().


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
